# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

feature_df = pd.DataFrame({
    "gsc_impressions": [1200, 850, 430, 2100, 650],
    "gsc_clicks": [48, 17, 9, 42, 13],
    "gsc_avg_position":[8.2, 14.5, 19.3, 11.7, 22.1]
})

feature_df["ctr"] = np.where(
    feature_df["gsc_impressions"] > 0,
    feature_df["gsc_clicks"] / feature_df["gsc_impressions"],
    0
)

feature_columns = [
    "gsc_impressions",
    "ctr",
    "gsc_avg_position"
]

x = feature_df[feature_columns].copy()

print("Feature columns:", feature_columns)
print(x)


Feature columns: ['gsc_impressions', 'ctr', 'gsc_avg_position']
   gsc_impressions      ctr  gsc_avg_position
0             1200  0.04000               8.2
1              850  0.02000              14.5
2              430  0.02093              19.3
3             2100  0.02000              11.7
4              650  0.02000              22.1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

I used hree search features:

-'gsc_impressions: how many times the page appeared in search.
-'ctr': how manpeople clicked compared with impressions.
'gsc_avg_position': the average position of the page insearch results.

If a value is missing i use simple fill values. I use 0 for impressions and CTR and 100 for average position.I do not use IDs as features. These features are avaliable before i make the ranking.








In [2]:
feature_notes = {
    "gsc_impressions": "Search impressions, fill missing with 0",
    "ctr": "Click_through rate, fill missing with 0",
    "gsc_avg_position": "Average search position, fill issing with 100"
}

feature_notes

{'gsc_impressions': 'Search impressions, fill missing with 0',
 'ctr': 'Click_through rate, fill missing with 0',
 'gsc_avg_position': 'Average search position, fill issing with 100'}

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the features for possible date leakage. I do not use content IDs as features. I also do not use trend direction or trend ct because they could contain information related to the result. The ranking only uses impressions, CTR and average search position. These values are avaliable while i create the ranking. I also use a time aware split so the earlier period and validation period are keot separate.

In [3]:
forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
    "is_declining_label"
}

used_features = set(feature_columns)

leaky_features = used_features.intersection(forbidden_features)

print("used feature:", feature_columns)
print("leaky features found:", list(leaky_features))

if len(leaky_features)==0:
  print("Leakage check passed: no forbidden features are used.")
else:
    print("Leakage warning: remove the features listed above.")

earlier_end = pd.Timestamp("2026-03-24")
validation_start = pd.Timestamp("2026-03-25")

print("Earlier period ends:", earlier_end.date())
print("Validation period starts:", validation_start.date())

if earlier_end < validation_start:
      print("Time split check passed: the priod do not overlap.")
else:
        print("Time split warning: the periods overlap.")

used feature: ['gsc_impressions', 'ctr', 'gsc_avg_position']
leaky features found: []
Leakage check passed: no forbidden features are used.
Earlier period ends: 2026-03-24
Validation period starts: 2026-03-25
Time split check passed: the priod do not overlap.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I exclude some fields from the features:
-'content_id': only used to identify the page not as a feature.
-'_client_id': only used for grouping not as a feature
-'trend_direction': excluded because it can contain information related to the result.
-'trend_pct': excluded because it can information related to the result.
-'is_declining_label': excluded because it is a label and should not be used as a feature.
I only kept the search features that i need for the ranking.

In [5]:
excluded_fields = {
    "content_id": "used only to identify the page, not as a prediction feature.",
    "_client_id": "Used only for grouping and splitting the date.",
    "trend_direction": "Excluded because it is related to the result we want to predict.",
    "trend_pct": "Excluded because it may give information about the target result.",
    "is_declining_label": "This is the target label, so it can not be used as an input feature."
}

for field, reason in excluded_fields.items():
  print(field, ":", reason)

content_id : used only to identify the page, not as a prediction feature.
_client_id : Used only for grouping and splitting the date.
trend_direction : Excluded because it is related to the result we want to predict.
trend_pct : Excluded because it may give information about the target result.
is_declining_label : This is the target label, so it can not be used as an input feature.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.